# SynTrustBench v0.5: main experiment

**Question.** Which whole benchmark gives verdicts you can trust, and what does each verdict rule out?

Every verdict on a gating dimension (fidelity, utility, privacy) carries a confidence statement, always per
dimension, never merged into one number:

| Verdict | Confidence statement | Source |
|---|---|---|
| Fail / Conditional | **Evidence strength** e = (1 + honest fit runs at least as extreme) / 101 | the gate paper's calibration machinery, refitted on STB's own 100 nulls and cited |
| Pass | **Limit of detection**: the smallest dose of each flaw caught with >= 95% probability (95% confidence) | new |
| Pass (privacy) | **Table-specific limit**: copies planted into the very table being judged | new |
| Generator | **Seed agreement**: modal verdict and k/5 | new |
| (all of the above) | **Validation**: fresh bases, claimed vs observed, transfer gap, head-to-head ratios | new |

Everything here follows `PREREGISTRATION_v0.5.md`. The code lives in `stb/` (one small module per job); this
notebook only calls it, stage by stage. Each stage writes one result file per job, so any cell can be
re-run and will skip finished work.

**Where to run.** Smoke mode runs anywhere in minutes and produces no evidence. The full run must happen on the
SJU server only (classifier signals differ slightly between Apple ARM and Linux x86; one results folder = one
platform).

## 0. Settings

In [1]:
import os; os.environ.update(STB_SMOKE='0', STB_DATASET='cdc', STB_WORKERS='16')
# Deviation (15 Nov 2026): AIM seeds 17 and 59 stopped after more than 60 h; AIM is reported with 3 of 5 seeds.
SKIP = ('DPAIM/17', 'DPAIM/59')
def keep(jobs):
    kept = [j for j in jobs if not any(s in repr(j) for s in SKIP)]
    print(f'{len(jobs) - len(kept)} jobs skipped for the stopped AIM seeds')
    return kept

In [2]:
# Settings come from environment variables so the run can be launched headless with run.sh;
# the defaults below apply when the notebook is opened by hand.
import os
from pathlib import Path


def physical_cores() -> int:
    """Physical cores (not hyper-threads), so WORKERS x THREADS never oversubscribes the machine."""
    import platform as _platform
    import subprocess
    try:
        if _platform.system() == 'Darwin':
            return int(subprocess.check_output(['sysctl', '-n', 'hw.physicalcpu']).strip())
        cores, phys, core = set(), None, None
        for line in open('/proc/cpuinfo').read().splitlines() + ['']:
            if line.startswith('physical id'):
                phys = line.split(':', 1)[1].strip()
            elif line.startswith('core id'):
                core = line.split(':', 1)[1].strip()
            elif not line.strip():
                if core is not None:
                    cores.add((phys, core))
                phys, core = None, None
        if cores:
            return len(cores)
    except Exception:  # noqa: BLE001 - fall back to a conservative guess
        pass
    return max(1, (os.cpu_count() or 2) // 2)


os.environ.setdefault('STB_FROZEN_CHECKOUT', str(Path.cwd() / 'vendor' / 'frozen_v030'))
SMOKE = os.environ.get('STB_SMOKE', '1') == '1'       # 1: toy sizes, any machine, no evidence. 0: the preregistered run.
DATASET = os.environ.get('STB_DATASET', 'cdc')        # 'cdc' (primary evidence) or 'uci' (replication)
THREADS = int(os.environ.get('STB_THREADS', 2 if SMOKE else 4))                    # threads per job
WORKERS = int(os.environ.get('STB_WORKERS', 2 if SMOKE else max(1, physical_cores() // THREADS)))  # parallel jobs
SMOKE_GENERATORS = ('GaussianCopula', 'CTGAN', 'TVAE', 'DPAIM')   # ('SmokeResample',) where SDV is not installed
assert DATASET in ('cdc', 'uci'), DATASET
print(f'SMOKE={SMOKE} DATASET={DATASET} WORKERS={WORKERS} THREADS={THREADS} (physical cores: {physical_cores()})')

SMOKE=False DATASET=cdc WORKERS=16 THREADS=4 (physical cores: 128)


In [3]:
import platform
from pathlib import Path

import pandas as pd

from stb import analysis, stages
from stb.bases import Workspace, prepare_cohort
from stb.operators import SINGLE_DOSE
from stb.plan import full_plan, smoke_plan
from stb.runner import run_jobs
from stb.store import write_manifest

CODE_ROOT = Path.cwd()
PLAN = smoke_plan(SMOKE_GENERATORS) if SMOKE else full_plan()
RESULTS_ROOT = CODE_ROOT / 'runs' / PLAN.name
ws = Workspace(DATASET, str(RESULTS_ROOT), PLAN.data_scale)
store = ws.store

if not SMOKE and (platform.system(), platform.machine()) != ('Linux', 'x86_64'):
    raise RuntimeError('The preregistered run is scored on the SJU server (Linux x86_64) only.')

manifest = write_manifest(store, PLAN, CODE_ROOT, CODE_ROOT / 'PREREGISTRATION_v0.5.md')
pd.set_option('display.max_colwidth', 80)
print(f'{PLAN.name} | {DATASET} | results in {ws.root}')
print('platform:', manifest['platform'], '| preregistration sha256:', (manifest['preregistration_sha256'] or '')[:12])

def run(jobs, label):
    """Run a stage; stop the notebook if any job failed (tracebacks are in <results>/failures/)."""
    print(f'{label}: {len(jobs)} jobs')
    summary = run_jobs(jobs, store, workers=WORKERS, threads=THREADS)
    if summary['failed']:
        raise RuntimeError(f"{summary['failed']} {label} failed; see {ws.root / 'failures'}")
    return summary

stb-v0.5-full | cdc | results in /home/sbukhari/SynTrustBench-Run/runs/stb-v0.5-full/cdc
platform: {'system': 'Linux', 'machine': 'x86_64', 'python': '3.12.16', 'cpu_count': 256} | preregistration sha256: bb0a9605f75c


## 1. Data
The frozen 40/40/20 person-disjoint splits (STB's own hash-verified copies). The held-out test set E is read
only when real generators are judged in section 4; null runs never touch it.

In [4]:
cohort = prepare_cohort(ws)
pd.DataFrame([{'role': k, 'sha256': v[:12]} for k, v in cohort['split_hashes'].items()])

,role,sha256
0,real_train,f568bb9ec7f7
1,calibration_pool,2e001f129574
2,real_test,4cac49278d2e


## 2. Honest null runs
Each run draws fresh pseudo-roles from the development data (STB's own seed salt, so no run coincides with a
gate-paper run), builds the real-data context, and scores its honest table. Ids are fixed in advance:

* **0-99 fit**: the calibration, all five dimensions (robustness included);
* **100-249 estimation bases** and **250-349 validation bases**: tables that flaws are planted into, with
  their dose-0 evidence.

Competitor statistics are scored on every run too: fit runs calibrate them, bases give their dose-0 rates.

In [5]:
run(stages.null_jobs(ws, PLAN), 'null runs')

null runs: 350 jobs


{'planned': 350, 'skipped_done': 350, 'ok': 0, 'failed': 0}

## 3. Calibration and evidence strength
STB refits the full-conformal maxT rule on its **own** fit runs (never the gate paper's frozen model). With
n = 100 and alpha = 0.05, r = floor(0.05 x 101) = 5: a dimension fails when its own maximum standardized signal
exceeds the 5th-largest fit-run maximum, and every Fail carries evidence e <= 5/101. The competitors are
calibrated by the same rule on the same runs.

In [6]:
calibration = stages.calibrate(ws, PLAN)
model = store.read('calibration', 'model')['model']
comp_model = store.read('calibration', 'competitors')['model']
calibration

{'sha256': '67aec01fefbde9ec329b59964f397e69cff9df8aca41ef69cf0d4eb9f19db86b',
 'n_fit': 100,
 'rejection_rank': {'0.025': 2, '0.05': 5, '0.1': 10},
 'blocked_dimensions': ['equity'],
 'excluded_signals': [{'dimension': 'equity',
   'metric': 'worst_group_retention',
   'family': 'diagnostic',
   'missing_run_ids': [3, 4, 8, 18, 22, 42, 45, 70, 72, 84, 85, 92],
   'reason': 'Required signal missing; calibrated dimension cannot Pass.'}]}

**H3 (a design check, not a discovery).** Honest Fails on bases never used for fitting, with exact one-sided
95% upper bounds; NotEvaluated is counted separately. H3 holds if the exact one-sided binomial test of the gate's
pooled Fail count against 5/101 does not reject at level 0.05 (row bases = 'all', dimension = 'gate').

In [7]:
analysis.null_false_alarms(store, model, PLAN.ids, PLAN.alpha)

,bases,dimension,fails,not_evaluated,runs,rate,exact_upper95,nominal_bound,binomial_p,h3_holds
0,estimation,fidelity,2,0,150,0.013333,0.041375,0.049505,NaN,None
1,estimation,utility,1,0,150,0.006667,0.031234,0.049505,NaN,None
2,estimation,privacy,0,0,150,0.000000,0.019773,0.049505,NaN,None
3,estimation,gate,3,0,150,0.020000,0.050877,0.049505,NaN,None
4,validation,fidelity,1,0,100,0.010000,0.046560,0.049505,NaN,None
5,validation,utility,3,0,100,0.030000,0.075711,0.049505,NaN,None
6,validation,privacy,0,0,100,0.000000,0.029513,0.049505,NaN,None
7,validation,gate,4,0,100,0.040000,0.089196,0.049505,NaN,None
8,all,fidelity,3,0,250,0.012000,0.030721,0.049505,NaN,None
9,all,utility,4,0,250,0.016000,0.036238,0.049505,NaN,None


## 4. Real generators x seeds
Each generator is fitted on the real training role and judged against the untouched held-out set, five seeds
each. The matrix generators also draw fresh samples that become bases for planted privacy flaws in section 6.

In [8]:
run(stages.main_jobs(ws), 'main roles')
run(keep(stages.generator_jobs(ws, PLAN)), 'generator fits')
gen = analysis.generator_table(store, model, PLAN.alpha)
agreement, stability = analysis.seed_summary(gen)
gen.pivot_table(index=['generator', 'seed'], columns='dimension', values='verdict', aggfunc='first')

main roles: 1 jobs
2 jobs skipped for the stopped AIM seeds
generator fits: 18 jobs


dimension                  equity fidelity privacy robustness utility
generator      seed                                                  
CTGAN          17            Fail     Fail    Pass       Fail    Fail
               29            Fail     Fail    Pass       Fail    Fail
               43            Fail     Fail    Pass       Pass    Fail
               59            Fail     Fail    Pass       Pass    Fail
               71            Fail     Fail    Pass       Pass    Fail
DPAIM          29    NotEvaluated     Fail    Pass       Fail    Fail
               43    NotEvaluated     Fail    Pass       Fail    Fail
               71    NotEvaluated     Fail    Pass       Fail    Fail
GaussianCopula 17    NotEvaluated     Fail    Pass       Fail    Fail
               29    NotEvaluated     Fail    Pass       Fail    Fail
               43    NotEvaluated     Fail    Pass       Fail    Fail
               59    NotEvaluated     Fail    Pass       Fail    Fail
               71    NotEvaluated     Fail    Pass       Fail    Fail
TVAE           17            Fail     Fail    Pass       Pass    Fail
               29            Fail     Fail    Pass       Pass    Fail
               43            Fail     Fail    Pass       Pass    Fail
               59            Fail     Fail    Pass       Fail    Fail
               71            Fail     Fail    Pass       Fail    Fail

In [9]:
display(agreement)
# H4: mean Kendall tau-b of generator rankings across seed pairs; unstable if below 2/3
pd.concat([stability, analysis.competitor_ranking_stability(store, comp_model)], ignore_index=True)

,generator,dimension,modal_verdict,agree,seeds,flag,counts
0,CTGAN,equity,Fail,5,5,stable,{'Fail': 5}
1,CTGAN,fidelity,Fail,5,5,stable,{'Fail': 5}
2,CTGAN,privacy,Pass,5,5,stable,{'Pass': 5}
3,CTGAN,robustness,Pass,3,5,seed-dependent,"{'Fail': 2, 'Pass': 3}"
4,CTGAN,utility,Fail,5,5,stable,{'Fail': 5}
5,DPAIM,equity,NotEvaluated,3,3,stable,{'NotEvaluated': 3}
6,DPAIM,fidelity,Fail,3,3,stable,{'Fail': 3}
7,DPAIM,privacy,Pass,3,3,stable,{'Pass': 3}
8,DPAIM,robustness,Fail,3,3,stable,{'Fail': 3}
9,DPAIM,utility,Fail,3,3,stable,{'Fail': 3}


,benchmark,dimension,mean_kendall_tau,pairs,unstable
0,STB,equity,1.000000,10,False
1,STB,fidelity,0.933333,10,False
2,STB,privacy,0.133333,10,True
3,STB,robustness,0.333333,10,True
4,STB,utility,0.066667,10,True
5,SDMetrics: sdmetrics_quality,fidelity,1.000000,10,False
6,SDMetrics: sdmetrics_dcr_baseline,privacy,0.600000,10,True
7,SDMetrics: sdmetrics_dcr_overfitting,privacy,0.524528,10,True
8,SDMetrics: sdmetrics_new_row_synthesis,privacy,0.533333,10,True


## 5. Limits of detection on honest tables (the primary matrix)
For every flaw family, one job plants the flaw at every dose of its grid into one estimation base (nested
doses, one seed), and re-scores only the flaw's target dimension. The official LoD is the lowest dose of an
unbroken run, from the top of the grid down, whose one-sided Clopper-Pearson lower bound is >= 95%. The probit
fit is descriptive only. Dose 0 gives the false-alarm rate.

In [10]:
run(stages.honest_lod_jobs(ws, PLAN), 'honest-matrix replicates')
outcomes = analysis.outcomes_frame(store)
lods = analysis.lod_summary(outcomes, PLAN.pi, PLAN.gamma)
lods[['matrix', 'family', 'target', 'eligible_bases', 'lod_fraction', 'lod_rows', 'status', 'lod_fit', 'false_alarm']]

honest-matrix replicates: 1350 jobs
  10/1350 done, 0.7 min elapsed
  20/1350 done, 1.3 min elapsed
  30/1350 done, 1.3 min elapsed
  40/1350 done, 1.9 min elapsed
  50/1350 done, 2.5 min elapsed
  60/1350 done, 2.5 min elapsed
  70/1350 done, 3.1 min elapsed
  80/1350 done, 3.5 min elapsed
  90/1350 done, 3.8 min elapsed
  100/1350 done, 4.1 min elapsed
  110/1350 done, 4.4 min elapsed
  120/1350 done, 4.4 min elapsed
  130/1350 done, 4.7 min elapsed
  140/1350 done, 5.0 min elapsed
  150/1350 done, 5.1 min elapsed
  160/1350 done, 5.5 min elapsed
  170/1350 done, 5.7 min elapsed
  180/1350 done, 6.0 min elapsed
  190/1350 done, 6.4 min elapsed
  200/1350 done, 6.6 min elapsed
  210/1350 done, 6.9 min elapsed
  220/1350 done, 7.3 min elapsed
  230/1350 done, 7.3 min elapsed
  240/1350 done, 7.7 min elapsed
  250/1350 done, 8.0 min elapsed
  260/1350 done, 8.2 min elapsed
  270/1350 done, 8.6 min elapsed
  280/1350 done, 8.9 min elapsed
  290/1350 done, 9.1 min elapsed
  300/1350 done,

/home/sbukhari/SynTrustBench-Run/.venv/lib/python3.12/site-packages/sdmetrics/reports/single_table/__init__.py:12: FutureWarning: The single table quality report is deprecated. Please use the QualityReport from 'sdmetrics.reports' instead.
  from sdmetrics.reports.single_table.quality_report import QualityReport
/home/sbukhari/SynTrustBench-Run/.venv/lib/python3.12/site-packages/sdmetrics/reports/single_table/__init__.py:12: FutureWarning: The single table quality report is deprecated. Please use the QualityReport from 'sdmetrics.reports' instead.
  from sdmetrics.reports.single_table.quality_report import QualityReport
/home/sbukhari/SynTrustBench-Run/.venv/lib/python3.12/site-packages/sdmetrics/reports/single_table/__init__.py:12: FutureWarning: The single table quality report is deprecated. Please use the QualityReport from 'sdmetrics.reports' instead.
  from sdmetrics.reports.single_table.quality_report import QualityReport
/home/sbukhari/SynTrustBench-Run/.venv/lib/python3.12/site

  750/1350 done, 28.4 min elapsed


/home/sbukhari/SynTrustBench-Run/.venv/lib/python3.12/site-packages/sdmetrics/reports/single_table/__init__.py:12: FutureWarning: The single table quality report is deprecated. Please use the QualityReport from 'sdmetrics.reports' instead.
  from sdmetrics.reports.single_table.quality_report import QualityReport
/home/sbukhari/SynTrustBench-Run/.venv/lib/python3.12/site-packages/sdmetrics/reports/single_table/__init__.py:12: FutureWarning: The single table quality report is deprecated. Please use the QualityReport from 'sdmetrics.reports' instead.
  from sdmetrics.reports.single_table.quality_report import QualityReport
/home/sbukhari/SynTrustBench-Run/.venv/lib/python3.12/site-packages/sdmetrics/reports/single_table/__init__.py:12: FutureWarning: The single table quality report is deprecated. Please use the QualityReport from 'sdmetrics.reports' instead.
  from sdmetrics.reports.single_table.quality_report import QualityReport
/home/sbukhari/SynTrustBench-Run/.venv/lib/python3.12/site

  760/1350 done, 29.1 min elapsed
  770/1350 done, 30.4 min elapsed
  780/1350 done, 31.0 min elapsed
  790/1350 done, 31.8 min elapsed
  800/1350 done, 33.1 min elapsed
  810/1350 done, 33.7 min elapsed
  820/1350 done, 34.5 min elapsed
  830/1350 done, 35.1 min elapsed
  840/1350 done, 36.0 min elapsed
  850/1350 done, 37.1 min elapsed
  860/1350 done, 37.7 min elapsed
  870/1350 done, 38.5 min elapsed
  880/1350 done, 39.7 min elapsed
  890/1350 done, 40.3 min elapsed
  900/1350 done, 41.1 min elapsed
  910/1350 done, 41.7 min elapsed
  920/1350 done, 42.7 min elapsed
  930/1350 done, 43.7 min elapsed
  940/1350 done, 44.3 min elapsed
  950/1350 done, 45.1 min elapsed
  960/1350 done, 45.9 min elapsed
  970/1350 done, 46.2 min elapsed
  980/1350 done, 47.0 min elapsed
  990/1350 done, 47.3 min elapsed
  1000/1350 done, 48.1 min elapsed
  1010/1350 done, 48.9 min elapsed
  1020/1350 done, 49.1 min elapsed
  1030/1350 done, 49.9 min elapsed
  1040/1350 done, 50.8 min elapsed
  1050/13

,matrix,family,target,eligible_bases,lod_fraction,lod_rows,status,lod_fit,false_alarm
0,honest,constraint_violation,fidelity,100,0.001,41.0,censored_at_lowest_dose,NaN,"{'hits': 1, 'trials': 100, 'rate': 0.01, 'wilson95': (0.0017674320641406505,..."
1,honest,copy_control,privacy,150,1.000,40589.0,censored_at_lowest_dose,NaN,None
2,honest,copy_outlier,privacy,150,0.020,812.0,certified,0.010323,"{'hits': 0, 'trials': 150, 'rate': 0.0, 'wilson95': (1.734723475976807e-18, ..."
3,honest,copy_random,privacy,150,0.020,812.0,certified,0.010558,"{'hits': 0, 'trials': 150, 'rate': 0.0, 'wilson95': (1.734723475976807e-18, ..."
4,honest,copy_test,privacy,150,NaN,NaN,not_detectable_in_range,NaN,None
5,honest,label_flip,utility,100,0.020,812.0,censored_at_lowest_dose,NaN,"{'hits': 1, 'trials': 100, 'rate': 0.01, 'wilson95': (0.0017674320641406505,..."
6,honest,marginal_shift,fidelity,100,0.100,4059.0,certified,0.066114,"{'hits': 1, 'trials': 100, 'rate': 0.01, 'wilson95': (0.0017674320641406505,..."
7,honest,near_copy,privacy,150,NaN,NaN,not_detectable_in_range,113.139830,"{'hits': 0, 'trials': 150, 'rate': 0.0, 'wilson95': (1.734723475976807e-18, ..."
8,honest,outcome_decouple,utility,100,0.100,4059.0,certified,0.051854,"{'hits': 1, 'trials': 100, 'rate': 0.01, 'wilson95': (0.0017674320641406505,..."
9,honest,shuffle,fidelity,100,0.300,12177.0,certified,0.272368,"{'hits': 1, 'trials': 100, 'rate': 0.01, 'wilson95': (0.0017674320641406505,..."


### 5b. Check of the target-only shortcut (amendment A6)
Section 5 re-scores only each flaw's target dimension and copies the other gating dimensions from the base's dose 0.
Because a candidate is pooled into every column's standardization, that shortcut could in principle change a
verdict. A fixed random 5% of the replicates of every family are re-scored in full (same planted tables, same
random streams), and the verdicts are compared. Report the agreement rate; if it is not essentially 100%, the
section 5 LoDs must be recomputed from full evaluations.

In [11]:
run(stages.a6_jobs(ws, PLAN), 'A6 full-evaluation check')
analysis.a6_agreement(store)

A6 full-evaluation check: 70 jobs
  10/70 done, 5.8 min elapsed
  20/70 done, 6.2 min elapsed
  30/70 done, 6.8 min elapsed
  40/70 done, 10.1 min elapsed
  50/70 done, 12.8 min elapsed
  60/70 done, 15.3 min elapsed
  70/70 done, 17.6 min elapsed


,family,tables,target_agree,off_target_agree,target_rate,target_wilson95,a6_holds
0,constraint_violation,35,35,10,1.000000,"(0.9010990076755959, 0.9999999999999999)",True
1,copy_control,8,8,8,1.000000,"(0.6755924351161198, 1.0)",True
2,copy_outlier,104,103,66,0.990385,"(0.9475322752920542, 0.9983006271503486)",True
3,copy_random,104,104,104,1.000000,"(0.9643786456275479, 0.9999999999999999)",True
4,copy_test,8,8,8,1.000000,"(0.6755924351161198, 1.0)",True
5,label_flip,35,35,8,1.000000,"(0.9010990076755959, 0.9999999999999999)",True
6,marginal_shift,35,35,10,1.000000,"(0.9010990076755959, 0.9999999999999999)",True
7,near_copy,104,103,87,0.990385,"(0.9475322752920542, 0.9983006271503486)",True
8,outcome_decouple,35,35,15,1.000000,"(0.9010990076755959, 0.9999999999999999)",True
9,shuffle,40,40,16,1.000000,"(0.9123783988027135, 1.0)",True


In [12]:
analysis.h2_attribution(store)   # H2, descriptive: target and co-firing rates on fully re-scored replicates

,family,target,dose,fraction,tables,fails_target,fails_utility,fails_privacy,fails_fidelity
0,constraint_violation,fidelity,0.001,0.001,5,1.0,0.0,0.0,NaN
1,constraint_violation,fidelity,0.005,0.005,5,1.0,1.0,0.0,NaN
2,constraint_violation,fidelity,0.01,0.010,5,1.0,1.0,0.0,NaN
3,constraint_violation,fidelity,0.02,0.020,5,1.0,1.0,0.0,NaN
4,constraint_violation,fidelity,0.05,0.050,5,1.0,1.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...
70,subgroup_degrade,equity,0.25,0.250,5,1.0,0.0,0.0,1.0
71,subgroup_degrade,equity,0.5,0.500,5,1.0,0.0,0.2,1.0
72,subgroup_degrade,equity,0.75,0.750,5,1.0,0.0,0.2,1.0
73,subgroup_degrade,equity,0.9,0.900,5,1.0,0.2,0.2,1.0


## 6. Do the limits transfer to generator output?
The same privacy flaws planted into fresh Gaussian copula and CTGAN samples. A base is eligible only if it passes
privacy at dose 0, so a detection means the planted flaw changed the verdict. The **transfer gap** is the observed
detection rate, in these messier tables, at the honest-matrix LoD.

In [13]:
run(stages.matrix_lod_jobs(ws, PLAN), 'generator-matrix replicates')
outcomes = analysis.outcomes_frame(store)
lods = analysis.lod_summary(outcomes, PLAN.pi, PLAN.gamma)
analysis.transfer_gap(outcomes, lods)

generator-matrix replicates: 900 jobs
  10/900 done, 1.0 min elapsed
  20/900 done, 2.1 min elapsed
  30/900 done, 2.1 min elapsed
  40/900 done, 3.1 min elapsed
  50/900 done, 4.1 min elapsed
  60/900 done, 4.2 min elapsed
  70/900 done, 5.1 min elapsed
  80/900 done, 5.3 min elapsed
  90/900 done, 6.2 min elapsed
  100/900 done, 7.1 min elapsed
  110/900 done, 7.2 min elapsed
  120/900 done, 8.2 min elapsed
  130/900 done, 9.1 min elapsed
  140/900 done, 9.3 min elapsed
  150/900 done, 10.2 min elapsed
  160/900 done, 10.6 min elapsed
  170/900 done, 11.7 min elapsed
  180/900 done, 12.6 min elapsed
  190/900 done, 13.0 min elapsed
  200/900 done, 14.1 min elapsed
  210/900 done, 15.0 min elapsed
  220/900 done, 15.4 min elapsed
  230/900 done, 16.3 min elapsed
  240/900 done, 16.9 min elapsed
  250/900 done, 17.8 min elapsed
  260/900 done, 18.7 min elapsed
  270/900 done, 19.1 min elapsed
  280/900 done, 20.2 min elapsed
  290/900 done, 21.1 min elapsed
  300/900 done, 21.5 min ela

,matrix,family,honest_lod,hits,trials,rate,wilson95
0,CTGAN,copy_outlier,0.02,1,150,0.006667,"(0.0011778013350174366, 0.03679283977481815)"
1,CTGAN,copy_random,0.02,0,150,0.000000,"(1.734723475976807e-18, 0.0249702443680766)"
2,GaussianCopula,copy_outlier,0.02,0,150,0.000000,"(1.734723475976807e-18, 0.0249702443680766)"
3,GaussianCopula,copy_random,0.02,0,150,0.000000,"(1.734723475976807e-18, 0.0249702443680766)"


## 7. Validation on fresh bases (confidence in the confidence)
Fresh validation bases (ids 250-349), fresh flaw seeds, at each honest LoD and its grid neighbours. The claim
holds if the Wilson interval at the LoD reaches 95%.

In [14]:
n_rows = int(outcomes.n_rows.iloc[0])
honest = lods[lods.matrix == 'honest'].set_index('family')
doses = {f: analysis.validation_doses(f, honest.loc[f].get('lod_fraction'), n_rows)
         for f in honest.index if f not in SINGLE_DOSE}
run(stages.validation_jobs(ws, PLAN, doses), 'validation replicates')
outcomes = analysis.outcomes_frame(store)
analysis.validation_summary(outcomes, lods, PLAN.pi)

validation replicates: 900 jobs
  10/900 done, 0.3 min elapsed
  20/900 done, 0.6 min elapsed
  30/900 done, 0.6 min elapsed
  40/900 done, 1.0 min elapsed
  50/900 done, 1.3 min elapsed
  60/900 done, 1.3 min elapsed
  70/900 done, 1.6 min elapsed
  80/900 done, 1.6 min elapsed
  90/900 done, 1.9 min elapsed
  100/900 done, 2.1 min elapsed
  110/900 done, 2.2 min elapsed
  120/900 done, 2.4 min elapsed
  130/900 done, 2.6 min elapsed
  140/900 done, 2.6 min elapsed
  150/900 done, 2.8 min elapsed
  160/900 done, 2.9 min elapsed
  170/900 done, 3.0 min elapsed
  180/900 done, 3.2 min elapsed
  190/900 done, 3.3 min elapsed
  200/900 done, 3.4 min elapsed
  210/900 done, 3.5 min elapsed
  220/900 done, 3.5 min elapsed
  230/900 done, 3.6 min elapsed
  240/900 done, 3.6 min elapsed
  250/900 done, 3.7 min elapsed
  260/900 done, 3.7 min elapsed
  270/900 done, 3.8 min elapsed
  280/900 done, 3.9 min elapsed
  290/900 done, 3.9 min elapsed
  300/900 done, 4.0 min elapsed
  310/900 done, 4

,family,dose,fraction,hits,trials,rate,wilson95,at_lod,pass
0,constraint_violation,0.001,0.001000,100,100,1.00,"(0.9630065017930143, 1.0)",True,True
1,constraint_violation,0.005,0.005000,100,100,1.00,"(0.9630065017930143, 1.0)",False,None
2,copy_outlier,0.01,0.010000,96,100,0.96,"(0.9016292856411208, 0.9843366960084523)",False,None
3,copy_outlier,0.02,0.020000,100,100,1.00,"(0.9630065017930143, 1.0)",True,True
4,copy_outlier,0.05,0.050000,100,100,1.00,"(0.9630065017930143, 1.0)",False,None
5,copy_random,0.01,0.010000,95,100,0.95,"(0.8882495307680808, 0.9784563208456319)",False,None
6,copy_random,0.02,0.020000,100,100,1.00,"(0.9630065017930143, 1.0)",True,True
7,copy_random,0.05,0.050000,100,100,1.00,"(0.9630065017930143, 1.0)",False,None
8,label_flip,0.02,0.020000,100,100,1.00,"(0.9630065017930143, 1.0)",True,True
9,label_flip,0.05,0.050000,100,100,1.00,"(0.9630065017930143, 1.0)",False,None


## 8. Table-specific privacy limit
For every generator table that passes privacy: copies planted into that very table, privacy re-scored.

In [15]:
run(keep(stages.table_lod_jobs(ws, PLAN)), 'table-specific replicates')
table_lods = analysis.table_lod_summary(store, PLAN.pi, PLAN.gamma)
table_lods

400 jobs skipped for the stopped AIM seeds
table-specific replicates: 3600 jobs
  10/3600 done, 0.4 min elapsed
  20/3600 done, 0.7 min elapsed
  30/3600 done, 0.7 min elapsed
  40/3600 done, 1.1 min elapsed
  50/3600 done, 1.4 min elapsed
  60/3600 done, 1.4 min elapsed
  70/3600 done, 1.8 min elapsed
  80/3600 done, 1.8 min elapsed
  90/3600 done, 2.1 min elapsed
  100/3600 done, 2.5 min elapsed
  110/3600 done, 2.7 min elapsed
  120/3600 done, 3.2 min elapsed
  130/3600 done, 3.6 min elapsed
  140/3600 done, 3.8 min elapsed
  150/3600 done, 4.3 min elapsed
  160/3600 done, 4.4 min elapsed
  170/3600 done, 4.9 min elapsed
  180/3600 done, 5.2 min elapsed
  190/3600 done, 5.5 min elapsed
  200/3600 done, 5.8 min elapsed
  210/3600 done, 6.1 min elapsed
  220/3600 done, 6.2 min elapsed
  230/3600 done, 6.5 min elapsed
  240/3600 done, 6.7 min elapsed
  250/3600 done, 6.9 min elapsed
  260/3600 done, 7.2 min elapsed
  270/3600 done, 7.3 min elapsed
  280/3600 done, 7.6 min elapsed
  290

,generator,seed,family,lod_fraction,lod_rows,status,steps,lod_fit,lod_fit_ci,false_alarm,curve
0,CTGAN,17,copy_outlier,0.10,5074,certified,"[{'dose': 0.25, 'hits': 100, 'trials': 100, 'rate': 1.0, 'cp_lower': 0.97048...",0.078373,"(0.07837339096971886, 0.07837339096971886)","{'hits': 0, 'trials': 100, 'rate': 0.0, 'wilson95': (3.469446951953614e-18, ...","[{'dose': '0', 'fraction': 0.0, 'rows': 0, 'hits': 0, 'trials': 100}, {'dose..."
1,CTGAN,17,copy_random,0.10,5074,certified,"[{'dose': 0.25, 'hits': 100, 'trials': 100, 'rate': 1.0, 'cp_lower': 0.97048...",0.066899,"(0.06550891993861953, 0.07837339096971886)","{'hits': 0, 'trials': 100, 'rate': 0.0, 'wilson95': (3.469446951953614e-18, ...","[{'dose': '0', 'fraction': 0.0, 'rows': 0, 'hits': 0, 'trials': 100}, {'dose..."
2,CTGAN,29,copy_outlier,0.10,5074,certified,"[{'dose': 0.25, 'hits': 100, 'trials': 100, 'rate': 1.0, 'cp_lower': 0.97048...",0.065505,"(0.0639507651473186, 0.07837339096971886)","{'hits': 0, 'trials': 100, 'rate': 0.0, 'wilson95': (3.469446951953614e-18, ...","[{'dose': '0', 'fraction': 0.0, 'rows': 0, 'hits': 0, 'trials': 100}, {'dose..."
3,CTGAN,29,copy_random,0.10,5074,certified,"[{'dose': 0.25, 'hits': 100, 'trials': 100, 'rate': 1.0, 'cp_lower': 0.97048...",0.066654,"(0.06550891993861953, 0.07837339096971886)","{'hits': 0, 'trials': 100, 'rate': 0.0, 'wilson95': (3.469446951953614e-18, ...","[{'dose': '0', 'fraction': 0.0, 'rows': 0, 'hits': 0, 'trials': 100}, {'dose..."
4,CTGAN,43,copy_outlier,0.05,2537,certified,"[{'dose': 0.25, 'hits': 100, 'trials': 100, 'rate': 1.0, 'cp_lower': 0.97048...",0.038756,"(0.03875574282568929, 0.03875574282568929)","{'hits': 0, 'trials': 100, 'rate': 0.0, 'wilson95': (3.469446951953614e-18, ...","[{'dose': '0', 'fraction': 0.0, 'rows': 0, 'hits': 0, 'trials': 100}, {'dose..."
5,CTGAN,43,copy_random,0.05,2537,certified,"[{'dose': 0.25, 'hits': 100, 'trials': 100, 'rate': 1.0, 'cp_lower': 0.97048...",0.038756,"(0.03875574282568929, 0.03875574282568929)","{'hits': 0, 'trials': 100, 'rate': 0.0, 'wilson95': (3.469446951953614e-18, ...","[{'dose': '0', 'fraction': 0.0, 'rows': 0, 'hits': 0, 'trials': 100}, {'dose..."
6,CTGAN,59,copy_outlier,0.05,2537,certified,"[{'dose': 0.25, 'hits': 100, 'trials': 100, 'rate': 1.0, 'cp_lower': 0.97048...",0.038756,"(0.03875574282568929, 0.03875574282568929)","{'hits': 0, 'trials': 100, 'rate': 0.0, 'wilson95': (3.469446951953614e-18, ...","[{'dose': '0', 'fraction': 0.0, 'rows': 0, 'hits': 0, 'trials': 100}, {'dose..."
7,CTGAN,59,copy_random,0.05,2537,certified,"[{'dose': 0.25, 'hits': 100, 'trials': 100, 'rate': 1.0, 'cp_lower': 0.97048...",0.038756,"(0.03875574282568929, 0.03875574282568929)","{'hits': 0, 'trials': 100, 'rate': 0.0, 'wilson95': (3.469446951953614e-18, ...","[{'dose': '0', 'fraction': 0.0, 'rows': 0, 'hits': 0, 'trials': 100}, {'dose..."
8,CTGAN,71,copy_outlier,0.05,2537,certified,"[{'dose': 0.25, 'hits': 100, 'trials': 100, 'rate': 1.0, 'cp_lower': 0.97048...",0.038756,"(0.03875574282568929, 0.03875574282568929)","{'hits': 0, 'trials': 100, 'rate': 0.0, 'wilson95': (3.469446951953614e-18, ...","[{'dose': '0', 'fraction': 0.0, 'rows': 0, 'hits': 0, 'trials': 100}, {'dose..."
9,CTGAN,71,copy_random,0.05,2537,certified,"[{'dose': 0.25, 'hits': 100, 'trials': 100, 'rate': 1.0, 'cp_lower': 0.97048...",0.034455,"(0.03100565679479385, 0.03875574282568929)","{'hits': 0, 'trials': 100, 'rate': 0.0, 'wilson95': (3.469446951953614e-18, ...","[{'dose': '0', 'fraction': 0.0, 'rows': 0, 'hits': 0, 'trials': 100}, {'dose..."


## 9. Head-to-head (H1)
Every benchmark statistic gets an LoD on the same planted tables and the same bases (the budgeted subset where
competitors were scored), each calibrated by the same conformal rule on the same fit runs. The H1 ratio compares
each competitor with STB's privacy-only variant (the matched setting); STB as shipped is shown too.

**Primary comparators** (they see the full table): NewRowSynthesis for privacy, the Quality Report for fidelity.
The DCR metrics see a 2,000-row subsample and are **secondary**, with that disclosed.
**Preregistered predictions:** NewRowSynthesis roughly ties STB on copy_random (it counts exact matches); STB's
advantage, if any, shows on near_copy and on CDC's chance-match correction; copy_test is missed by every
benchmark (section 5 detection rate), a shared blind spot, not a win.

In [16]:
h2h = analysis.head_to_head(outcomes, model, comp_model, PLAN.alpha, PLAN.pi, PLAN.gamma)
h2h

,family,benchmark,role,bases,lod_fraction,lod_rows,status,false_alarm,ratio_vs_stb
0,constraint_violation,"STB (gate, as shipped)",stb,100,0.001,41.0,censored_at_lowest_dose,"{'hits': 1, 'trials': 100, 'rate': 0.01, 'wilson95': (0.0017674320641406505,...",1.0
1,constraint_violation,STB (fidelity only),stb,100,0.001,41.0,censored_at_lowest_dose,"{'hits': 2, 'trials': 100, 'rate': 0.02, 'wilson95': (0.00550196755016235, 0...",1.0
2,copy_control,"STB (gate, as shipped)",stb,150,1.000,40589.0,censored_at_lowest_dose,None,1.0
3,copy_control,STB (privacy only),stb,150,1.000,40589.0,censored_at_lowest_dose,None,1.0
4,copy_outlier,"STB (gate, as shipped)",stb,150,0.020,812.0,certified,"{'hits': 0, 'trials': 150, 'rate': 0.0, 'wilson95': (1.734723475976807e-18, ...",2.0
5,copy_outlier,STB (privacy only),stb,150,0.010,406.0,certified,"{'hits': 3, 'trials': 150, 'rate': 0.02, 'wilson95': (0.006824751322967459, ...",1.0
6,copy_random,"STB (gate, as shipped)",stb,100,0.020,812.0,certified,"{'hits': 0, 'trials': 100, 'rate': 0.0, 'wilson95': (3.469446951953614e-18, ...",2.0
7,copy_random,STB (privacy only),stb,100,0.010,406.0,certified,"{'hits': 2, 'trials': 100, 'rate': 0.02, 'wilson95': (0.00550196755016235, 0...",1.0
8,copy_random,SDMetrics: sdmetrics_new_row_synthesis,primary,100,0.005,203.0,certified,"{'hits': 6, 'trials': 100, 'rate': 0.06, 'wilson95': (0.027786123963188192, ...",0.5
9,copy_test,"STB (gate, as shipped)",stb,150,NaN,NaN,not_detectable_in_range,None,NaN


### 9b. The DCR comparison (supporting figure, detection rates only)
SDMetrics' DCR metrics are the field's standard privacy check but too slow for the full grid. They are scored at
four doses of copy_random and near_copy on the first 30 estimation bases. DCR's own subsampling shrinks both the
released table and the training table to 2,000 rows, so STB is also scored at that budget (and calibrated on the
same kind of samples of the fit runs). Three curves per family: DCR, STB at DCR's budget, STB as shipped.

In [17]:
run(stages.dcr_null_jobs(ws, PLAN), 'STB at DCR budget: null runs')
stages.calibrate_budget(ws, PLAN)
budget_model = store.read('calibration', 'stb_budget')['model']
run(stages.dcr_jobs(ws, PLAN), 'DCR comparison')
analysis.dcr_comparison(store, comp_model, model, budget_model, PLAN.alpha)

STB at DCR budget: null runs: 130 jobs
  10/130 done, 0.0 min elapsed
  20/130 done, 0.0 min elapsed
  30/130 done, 0.0 min elapsed
  40/130 done, 0.1 min elapsed
  50/130 done, 0.1 min elapsed
  60/130 done, 0.1 min elapsed
  70/130 done, 0.1 min elapsed
  80/130 done, 0.1 min elapsed
  90/130 done, 0.1 min elapsed
  100/130 done, 0.1 min elapsed
  110/130 done, 0.1 min elapsed
  120/130 done, 0.1 min elapsed
  130/130 done, 0.1 min elapsed
DCR comparison: 60 jobs
  10/60 done, 4.4 min elapsed
  20/60 done, 8.7 min elapsed
  30/60 done, 8.9 min elapsed
  40/60 done, 14.1 min elapsed
  50/60 done, 19.2 min elapsed
  60/60 done, 19.6 min elapsed


,family,benchmark,dose,fraction,hits,trials,rate,wilson95
0,copy_random,SDMetrics: sdmetrics_dcr_baseline,0,0.00,2,30,0.066667,"(0.018477023791270378, 0.2132345836261692)"
1,copy_random,SDMetrics: sdmetrics_dcr_baseline,0.01,0.01,1,30,0.033333,"(0.005908590381612455, 0.16670390991409173)"
2,copy_random,SDMetrics: sdmetrics_dcr_baseline,0.05,0.05,4,30,0.133333,"(0.053096554840547455, 0.296813266820363)"
3,copy_random,SDMetrics: sdmetrics_dcr_baseline,0.25,0.25,2,30,0.066667,"(0.018477023791270378, 0.2132345836261692)"
4,copy_random,SDMetrics: sdmetrics_dcr_overfitting,0,0.00,2,30,0.066667,"(0.018477023791270378, 0.2132345836261692)"
5,copy_random,SDMetrics: sdmetrics_dcr_overfitting,0.01,0.01,2,30,0.066667,"(0.018477023791270378, 0.2132345836261692)"
6,copy_random,SDMetrics: sdmetrics_dcr_overfitting,0.05,0.05,2,30,0.066667,"(0.018477023791270378, 0.2132345836261692)"
7,copy_random,SDMetrics: sdmetrics_dcr_overfitting,0.25,0.25,2,30,0.066667,"(0.018477023791270378, 0.2132345836261692)"
8,copy_random,"STB (as shipped, full table)",0,0.00,0,30,0.000000,"(0.0, 0.11351339317396876)"
9,copy_random,"STB (as shipped, full table)",0.01,0.01,30,30,1.000000,"(0.8864866068260312, 0.9999999999999999)"


## 10. Equity admission (preregistered rule)
Equity is promoted to gating on this dataset only if it is evaluable everywhere, shows no excess honest
non-acceptance, and has an official LoD. If it is admitted, every verdict and LoD on this dataset is recomputed
from the saved signals with equity in the gate family (both versions reported).

In [18]:
analysis.equity_admission(store, model, lods, PLAN.ids, PLAN.alpha)

{'evaluable': False,
 'honest_fails': 8,
 'bases': 150,
 'binomial_p': 0.47723288686303106,
 'no_excess': True,
 'official_lod': True,
 'admitted': False}

## 11. Per-dimension records (spec section 10)
One record per generator table and gating dimension: verdict, evidence strength, the limits of detection for
the flaw families that target the dimension, the table-specific privacy limit, and seed agreement.

In [19]:
records = analysis.dimension_records(DATASET, gen, lods, table_lods, agreement)
store.write('records', 'dimension_records', {'records': records})
for r in records[:6]:
    print(r['text'], '\n')

**Fidelity: Fail** (5/5 seeds) More extreme than all 100 honest calibration tables. 

**Utility: Fail** (5/5 seeds) More extreme than all 100 honest calibration tables. 

**Privacy: Pass** (5/5 seeds) 97 of 100 honest calibration tables were at least as extreme. Rules out copying of rare (outlying) training records at >= 2% of rows (812.0 rows); Rules out copying of training records at >= 2% of rows (812.0 rows); Rules out near-copies of training records at >= nan% of rows (nan rows), each detected with >= 95% probability (95% confidence). For this table: Rules out copying of rare (outlying) training records at >= 10% of rows (5,074 rows). For this table: Rules out copying of training records at >= 10% of rows (5,074 rows). 

**Fidelity: Fail** (5/5 seeds) More extreme than all 100 honest calibration tables. 

**Utility: Fail** (5/5 seeds) More extreme than all 100 honest calibration tables. 

**Privacy: Pass** (5/5 seeds) 100 of 100 honest calibration tables were at least as extreme. 

## 12. Figures
To be added once real results exist: dose-response curves (Figure 1), claimed vs observed detection
(Figure 2), seed stability (Figure 3). They will read only the saved result files.